# Machine Learning Fundamentals

This section introduces the fundamental concepts of machine learning and deep learning that form the foundation for our magnetic field prediction approach. We explore the mathematical foundations, optimization theory, and generalization principles that enable deep learning models to learn complex relationships in electromagnetic systems.

## Learning Objectives

After completing this notebook, you will understand:

- **Neural Network Architecture**: From basic perceptrons to deep networks
- **Optimization Theory**: Gradient descent and backpropagation algorithms
- **Generalization**: Overfitting, regularization, and model selection
- **Loss Functions**: Designing objectives for regression tasks
- **Training Dynamics**: Learning rates, batch sizes, and convergence

## What is Machine Learning?

**Machine Learning** is a subfield of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed {cite}`bishop2006pattern,goodfellow2016deep`. Unlike traditional algorithmic approaches that encode explicit rules, machine learning algorithms discover relationships by optimizing model parameters to minimize prediction errors on training examples.

**Three main paradigms:**

1. **Supervised Learning**: Learn mapping from inputs to outputs given labeled training examples
   - Example: Predict house prices from features (size, location) using historical sales data
   
2. **Unsupervised Learning**: Discover structure in unlabeled data
   - Example: Cluster similar customers based on purchasing patterns

3. **Reinforcement Learning**: Learn actions to maximize rewards through trial and error
   - Example: Game-playing agents learning strategies through self-play

**This chapter focuses on supervised learning for regression**, where we train neural networks to predict continuous-valued outputs (magnetic flux density values) from geometric inputs.

:::{note}
**For EM Students**  
In supervised learning for field prediction:
- **Input**: Motor geometry parameters (slot width, magnet thickness, airgap length, etc.)
- **Output**: Magnetic flux density distribution (256×256 pixel field map)
- **Training data**: Thousands of FEA simulations providing input-output pairs
- **Goal**: Learn the mapping from geometry → field distribution to replace expensive FEA
:::

## What is Deep Learning?

**Deep Learning** is a subset of machine learning using neural networks with multiple layers {cite}`goodfellow2016deep,lecun2015deep`. The "depth" refers to the number of successive transformations:

$$\text{Input} \xrightarrow{\text{Layer 1}} \text{Features}_1 \xrightarrow{\text{Layer 2}} \text{Features}_2 \xrightarrow{\cdots} \xrightarrow{\text{Layer L}} \text{Output}$$

**Key advantage**: Deep networks learn **hierarchical representations** at multiple scales:
- **Early layers**: Detect local patterns (edges, textures in images; material boundaries in EM)
- **Middle layers**: Combine local features (object parts in images; flux zones in EM)  
- **Late layers**: Integrate global context (whole objects in images; complete field patterns in EM)

**Why deep learning for electromagnetic field prediction?**
- Traditional methods (polynomial regression, Kriging) struggle with high-dimensional spatial outputs (65,536 values for 256×256 field map)
- Convolutional neural networks (CNNs) naturally handle spatial data
- CNNs have demonstrated superior performance for field distribution prediction {cite}`khan2019deep`

:::{note}
**For EM Students**  
Deep CNNs learn electromagnetic relationships at multiple scales:
- **Layer 1-2**: Local field gradients at material boundaries (iron-air interface, magnet edges)
- **Layer 3-4**: Regional flux patterns (tooth saturation, airgap field distribution)
- **Layer 5-6**: Global field structure respecting Maxwell's equations (flux continuity, source-field relationships)

This hierarchical learning is why CNNs outperform shallow networks for field prediction.
:::

## Historical Context and Evolution

:::{note}
**For EM Students**  
While this section covers general machine learning history, these developments directly enabled electromagnetic field prediction. Neural network foundations, optimization algorithms trained on FEA simulations, and regularization techniques all ensure models generalize to new motor geometries. Understanding this evolution provides context for the CNN architectures and training methods developed in Sections 4-5.
:::

### From Theory to Practice

Deep learning foundations date back to the 1940s {cite}`goodfellow2016deep`:

- **1943**: McCulloch-Pitts neuron - First mathematical model {cite}`mcculloch1943logical`
- **1958**: Perceptron - Rosenblatt's learning algorithm {cite}`rosenblatt1958perceptron`
- **1986**: Backpropagation rediscovered and popularized {cite}`rumelhart1988learning`
- **1998**: LeNet-5 - First successful CNN for digit recognition {cite}`lecun1998gradient`
- **2012**: AlexNet - ImageNet breakthrough enabling modern deep learning {cite}`krizhevsky2012imagenet`

### Key Enabling Factors for Deep Learning Success

1. **Computational Power**: GPUs accelerating training 10-100× {cite}`goodfellow2016deep`
2. **Big Data**: Millions of labeled examples for supervised learning
3. **Algorithmic Advances**: ReLU activations {cite}`nair2010rectified`, batch normalization {cite}`ioffe2015batch`, adaptive optimizers {cite}`kingma2014adam`
4. **Software Ecosystem**: TensorFlow, PyTorch enabling rapid prototyping

## Neural Network Fundamentals

### 1. Basic Building Blocks

#### Artificial Neuron

The fundamental unit of a neural network is the **artificial neuron**, which computes:

$$y = f(\mathbf{w}^T\mathbf{x} + b)$$

where:
- $\mathbf{x}$ is the input vector
- $\mathbf{w}$ are the weights
- $b$ is the bias
- $f$ is the activation function

:::{note}
**For EM Students**  
For field prediction, input $\mathbf{x}$ could represent geometry parameters (slot width, magnet thickness), weights $\mathbf{w}$ learn the electromagnetic relationships, and output $y$ predicts flux density at a spatial location.
:::

#### Activation Functions

Common activation functions enable non-linear modeling {cite}`goodfellow2016deep,nair2010rectified`:

**ReLU (Rectified Linear Unit)** {cite}`nair2010rectified`:
$$f(x) = \max(0, x)$$
- Advantages: No vanishing gradient for positive inputs, computationally efficient
- Used in: Most modern CNNs including our field prediction networks

**Sigmoid**:
$$f(x) = \frac{1}{1 + e^{-x}}$$
- Advantages: Smooth, bounded output [0,1]
- Disadvantages: Vanishing gradient problem, not zero-centered

**Tanh**:
$$f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$
- Advantages: Zero-centered, smooth
- Disadvantages: Vanishing gradient problem

:::{note}
Detailed activation function analysis with visualizations and derivatives is provided in **Section 4a (ANN Fundamentals)**.  
Here we focus on conceptual understanding and selection criteria.
:::

### 2. Network Architectures

#### Feedforward Neural Networks

A feedforward network with $L$ layers computes:

$$\mathbf{h}^{(l)} = f^{(l)}(\mathbf{W}^{(l)}\mathbf{h}^{(l-1)} + \mathbf{b}^{(l)})$$

for $l = 1, 2, \ldots, L$, with $\mathbf{h}^{(0)} = \mathbf{x}$ (input) and $\mathbf{y} = \mathbf{h}^{(L)}$ (output).

#### Deep vs. Shallow Networks

**Universal Approximation Theorem** {cite}`hornik1989multilayer`: A feedforward network with a single hidden layer can approximate any continuous function on a compact domain.

However, **deep networks** (multiple hidden layers) offer advantages {cite}`goodfellow2016deep,bengio2007scaling`:
- **Exponential expressivity**: Fewer parameters needed for same function complexity
- **Hierarchical feature learning**: Learn representations at multiple scales (critical for capturing both local saturation effects and global flux patterns in EM devices)
- **Better generalization**: Implicit regularization through depth

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.patches as patches
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Machine Learning Fundamentals Framework Initialized")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
# Brief activation function demonstration (detailed analysis in Section 4a)

def demonstrate_activation_concepts():
    """
    Brief visualization of key activation function properties.
    Full analysis with derivatives in Section 4a (ANN Fundamentals).
    """

    x = np.linspace(-3, 3, 500)

    # Define key activations
    def relu(x):
        return np.maximum(0, x)

    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    def tanh(x):
        return np.tanh(x)

    # Single comparison plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    fig.suptitle('Activation Functions: Key Properties Comparison',
                 fontsize=14, fontweight='bold')

    ax.plot(x, relu(x), linewidth=3, label='ReLU', color='#FF6B6B')
    ax.plot(x, sigmoid(x), linewidth=3, label='Sigmoid', color='#4ECDC4')
    ax.plot(x, tanh(x), linewidth=3, label='Tanh', color='#45B7D1')

    ax.set_xlabel('Input', fontsize=12)
    ax.set_ylabel('Output', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='k', linestyle='-', alpha=0.3)

    # Add annotations for key properties
    ax.text(-2.5, 2.2, '✓ ReLU: No vanishing gradient\n   for positive inputs',
            bbox=dict(boxstyle="round,pad=0.5", facecolor='yellow', alpha=0.3),
            fontsize=10)

    ax.text(1.2, 0.3, '⚠ Sigmoid/Tanh: Vanishing\n   gradient problem',
            bbox=dict(boxstyle="round,pad=0.5", facecolor='orange', alpha=0.3),
            fontsize=10)

    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("Activation Function Selection for EM Field Prediction")
    print("="*60)
    print("\n[OK] ReLU (Used in this work):")
    print("    - Fast training due to non-saturating gradient")
    print("    - Sparse activations beneficial for field sparsity")
    print("    - Dead neuron problem mitigated by proper initialization")
    print("\n[INFO] Sigmoid:")
    print("    - Historically used, now mostly replaced by ReLU")
    print("    - Can cause vanishing gradients in deep networks")
    print("\n[INFO] Tanh:")
    print("    - Zero-centered (slight advantage over sigmoid)")
    print("    - Still suffers from vanishing gradient")
    print("\n→ See Section 4a for complete analysis with derivatives")

demonstrate_activation_concepts()

## Optimization and Training

### 1. Loss Functions for Regression

For magnetic field prediction, we use regression loss functions that quantify the difference between predicted and target field distributions. The choice of loss function affects both optimization dynamics and the characteristics of learned models.

#### Mean Squared Error (MSE)

The **Mean Squared Error** is the most widely used loss function for regression tasks:

$$\mathcal{L}_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Term definitions:**
- $n$: Number of samples in the training batch (or total dataset)
- $y_i$: True target value for the $i$-th sample (ground truth from FEA)
- $\hat{y}_i$: Predicted value from the neural network for the $i$-th sample
- $(y_i - \hat{y}_i)$: Prediction error (residual) for sample $i$

**Properties:**
- **Quadratic penalty**: Errors are squared, so large deviations are penalized more heavily than small ones
- **Differentiable everywhere**: Smooth gradient enables efficient optimization via gradient descent
- **Sensitivity to outliers**: Large errors dominate the loss due to squaring
- **Units**: If $y$ has units of Tesla (T), loss has units of T²

**Multi-output aggregation for spatial field maps:**

For electromagnetic field prediction, each training sample consists of a complete spatial distribution with thousands of output values. Consider a 256×256 pixel field map:

$$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{j=1}^{N} \frac{1}{H \cdot W} \sum_{h=1}^{H} \sum_{w=1}^{W} \left( y_{j,h,w} - \hat{y}_{j,h,w} \right)^2$$

where:
- $N$: Batch size (number of different motor geometries)
- $H \times W$: Spatial dimensions (e.g., 256×256 pixels)
- $y_{j,h,w}$: True magnetic flux density at pixel $(h,w)$ for geometry $j$
- $\hat{y}_{j,h,w}$: CNN-predicted flux density at pixel $(h,w)$ for geometry $j$

This formulation averages the squared error across all spatial locations and all samples in the batch, treating each pixel's prediction with equal weight.

:::{note}
**For EM Students**  
For a 256×256 field map, each CNN prediction generates 65,536 output values. MSE aggregates errors across all pixels:
- **High-flux regions** (magnets: 1-2 T, airgap: 0.5-1 T): Larger absolute errors, dominate loss
- **Low-flux regions** (far field: 0.001-0.1 T): Smaller absolute errors, minimal contribution
- **Design implication**: MSE naturally emphasizes accuracy where it matters most for torque/force calculation

**Alternative**: Some works use **normalized MSE** by dividing by the maximum flux density to balance contributions across different motor designs with varying field strengths.
:::

#### Mean Absolute Error (MAE)

The **Mean Absolute Error** uses absolute value instead of squaring:

$$\mathcal{L}_{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Term definitions:**
- Same as MSE, but using absolute value $|\cdot|$ instead of square $(\cdot)^2$

**Properties:**
- **Linear penalty**: All errors weighted equally regardless of magnitude
- **Robust to outliers**: Large errors don't dominate the loss as strongly as in MSE
- **Not differentiable at zero**: Gradient undefined at $y_i = \hat{y}_i$ (subgradient methods used in practice)
- **Units**: Same units as target variable (Tesla for flux density)

**Comparison with MSE:**
- **MSE**: Penalizes large errors more → encourages predictions closer to true values in high-variance regions
- **MAE**: Treats all errors equally → less sensitive to anomalous FEA solutions or mesh artifacts
- **For EM field prediction**: MSE is generally preferred because accurate prediction in high-flux regions (magnets, airgap) is more critical for downstream calculations (torque, losses) than accuracy in low-flux regions

#### Huber Loss

The **Huber loss** combines the advantages of MSE and MAE through a piecewise formulation:

$$\mathcal{L}_{Huber}(y, \hat{y}; \delta) = \frac{1}{n} \sum_{i=1}^{n} L_\delta(y_i - \hat{y}_i)$$

where:

$$L_\delta(r) = \begin{cases} 
\frac{1}{2}r^2 & \text{if } |r| \leq \delta \\
\delta \left( |r| - \frac{\delta}{2} \right) & \text{if } |r| > \delta 
\end{cases}$$

**Term definitions:**
- $r = y_i - \hat{y}_i$: Residual (prediction error)
- $\delta > 0$: Threshold parameter (hyperparameter)
- For $|r| \leq \delta$: Quadratic loss (behaves like MSE)
- For $|r| > \delta$: Linear loss (behaves like MAE)

**Properties:**
- **Smooth transition**: Continuously differentiable everywhere (unlike MAE)
- **Balanced sensitivity**: Quadratic for small errors (fast convergence), linear for large errors (outlier robustness)
- **Hyperparameter $\delta$**: Controls transition point
  - Small $\delta$ (e.g., 0.1): More MAE-like, robust to outliers
  - Large $\delta$ (e.g., 1.0): More MSE-like, emphasizes accuracy

**When to use Huber loss in EM applications:**
- Training data contains occasional poor-quality FEA solutions (mesh convergence issues, solver tolerance violations)
- Field distributions have extreme local variations (sharp corners, thin airgaps) that may cause numerical artifacts
- Hybrid approach desired: accurate in typical regions, robust to anomalies

**Practical note**: Most EM field prediction studies use MSE due to its simplicity and the fact that FEA training data is generally high-quality. Huber loss is worth exploring if encountering training instability or outlier sensitivity.

:::{tip}
**Loss Function Selection for EM Field Prediction**  
**MSE** (used in this work): Standard choice, emphasizes high-flux accuracy  
**MAE**: Consider if FEA data contains outliers or artifacts  
**Huber**: Best of both worlds, but requires tuning $\delta$ hyperparameter  

In Sections 4c-4d, we demonstrate that MSE achieves <1% normalized error for field prediction across all three electromagnetic case studies (coil, transformer, motor).
:::

### 2. Gradient Descent Variants

Optimization algorithms iteratively update neural network weights to minimize the loss function. The general form is:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \cdot \mathbf{g}_t(\mathbf{w}_t)$$

where:
- $\mathbf{w}_t$: Weight vector at iteration $t$
- $\eta_t > 0$: Learning rate at iteration $t$ (step size)
- $\mathbf{g}_t$: Update direction (varies by algorithm)

The choice of optimizer determines convergence speed, stability, and final solution quality.

#### Stochastic Gradient Descent (SGD) {cite}`goodfellow2016deep`

The foundational optimization algorithm:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)$$

**Term definitions:**
- $\nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)$: Gradient of loss function with respect to weights
- $\eta$: Learning rate (constant, typical values: $10^{-3}$ to $10^{-1}$)

**Properties:**
- **Simple**: Moves directly opposite to gradient direction
- **Noisy convergence**: Uses mini-batches, so gradient is stochastic estimate of true gradient
- **Fixed learning rate**: May converge slowly or oscillate near minima
- **No momentum**: Can get stuck in poor local minima or saddle points

**Disadvantages for deep learning:**
- Slow convergence, especially in regions with small gradients
- Sensitive to learning rate choice (too large: diverges, too small: slow)
- Poor performance in high-curvature loss landscapes (common in deep networks)

#### SGD with Momentum {cite}`polyak1964some`

Adds velocity term to accelerate convergence:

$$\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \mathbf{v}_{t+1}$$

**Term definitions:**
- $\mathbf{v}_t$: Velocity vector (exponentially weighted average of past gradients)
- $\beta \in [0, 1)$: Momentum coefficient (typical value: 0.9)
  - $\beta = 0$: Reduces to standard SGD
  - $\beta \to 1$: Stronger memory of past gradients
- First iteration: $\mathbf{v}_0 = \mathbf{0}$ (zero initialization)

**Physical analogy**: Ball rolling down a hill
- Gradient provides force (acceleration)
- Momentum causes ball to continue moving even if gradient is small
- Overshoots local minima with enough velocity

**Advantages:**
- **Accelerates in relevant directions**: Gradients that consistently point the same direction accumulate
- **Dampens oscillations**: Opposing gradients cancel out in velocity
- **Escapes shallow local minima**: Momentum can carry optimization through small barriers

#### RMSprop {cite}`tieleman2012rmsprop`

Adapts learning rate individually for each parameter based on recent gradient magnitudes:

$$\mathbf{v}_t = \gamma \mathbf{v}_{t-1} + (1-\gamma) \left( \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t) \right)^2$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{\mathbf{v}_t + \epsilon}} \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)$$

**Term definitions:**
- $\mathbf{v}_t$: Exponentially weighted moving average of squared gradients (element-wise)
- $\gamma \in [0, 1)$: Decay rate (typical value: 0.9 or 0.99)
- $\epsilon > 0$: Small constant for numerical stability (typical: $10^{-8}$)
- $\sqrt{\mathbf{v}_t + \epsilon}$: Element-wise square root, acts as adaptive learning rate
- Division and square operations are **element-wise** (per parameter)

**Key insight**: Normalize gradient by root-mean-square (RMS) of recent gradients
- Parameters with large typical gradients → smaller effective learning rate
- Parameters with small typical gradients → larger effective learning rate

**Advantages:**
- **Adaptive learning rates**: Different parameters trained at different speeds
- **Handles sparse gradients**: Useful when some features are rarely activated
- **Non-stationary objectives**: Adjusts to changing gradient statistics during training

**For EM applications**: Useful when field values span multiple orders of magnitude (0.001 T far field, 2 T in magnets)

#### Adam Optimizer {cite}`kingma2014adam`

**Adaptive Moment Estimation** combines momentum (first moment) and RMSprop (second moment):

**Algorithm:**

1. Compute gradient: $\mathbf{g}_t = \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)$

2. Update biased first moment estimate (momentum):
   $$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1-\beta_1) \mathbf{g}_t$$

3. Update biased second moment estimate (squared gradients):
   $$\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1-\beta_2) \mathbf{g}_t^2$$

4. Compute bias-corrected first moment:
   $$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1-\beta_1^t}$$

5. Compute bias-corrected second moment:
   $$\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1-\beta_2^t}$$

6. Update weights:
   $$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$$

**Term definitions:**
- $\mathbf{g}_t$: Gradient at iteration $t$ (vector, one element per weight)
- $\mathbf{m}_t$: First moment estimate (mean of gradients, like momentum)
- $\mathbf{v}_t$: Second moment estimate (uncentered variance of gradients)
- $\beta_1 \in [0, 1)$: Exponential decay rate for first moment (typical: 0.9)
- $\beta_2 \in [0, 1)$: Exponential decay rate for second moment (typical: 0.999)
- $\eta > 0$: Learning rate (typical: $10^{-3}$ to $10^{-4}$ for CNNs)
- $\epsilon > 0$: Small constant preventing division by zero (typical: $10^{-8}$)
- $\hat{\mathbf{m}}_t, \hat{\mathbf{v}}_t$: Bias-corrected moment estimates
- $t$: Iteration counter (starts at 1)

**Why bias correction?**
Initialize $\mathbf{m}_0 = \mathbf{0}$ and $\mathbf{v}_0 = \mathbf{0}$. In early iterations:
- $\mathbf{m}_t$ and $\mathbf{v}_t$ are biased toward zero (due to initialization)
- Bias correction terms $(1-\beta_1^t)^{-1}$ and $(1-\beta_2^t)^{-1}$ compensate
- As $t \to \infty$: $\beta_1^t \to 0$ and $\beta_2^t \to 0$, so $\hat{\mathbf{m}}_t \approx \mathbf{m}_t$ and $\hat{\mathbf{v}}_t \approx \mathbf{v}_t$

**Properties:**
- **Combines momentum and adaptive learning rates**: Benefits of both SGD+Momentum and RMSprop
- **Efficient**: Low memory overhead (stores two vectors: $\mathbf{m}$ and $\mathbf{v}$)
- **Invariant to gradient rescaling**: Normalizes by gradient magnitude
- **Well-suited for sparse gradients**: Handles varying gradient statistics

**Default hyperparameters (recommended starting point):**
- Learning rate $\eta = 10^{-3}$ (most common)
- $\beta_1 = 0.9$
- $\beta_2 = 0.999$
- $\epsilon = 10^{-8}$

**For EM field prediction CNNs (used in this work):**
- Adam consistently outperforms other optimizers in our experiments (Section 4c)
- Converges 2-5× faster than SGD+Momentum on FEA datasets
- Robust to initial learning rate choice
- Handles varying field magnitudes (0.001-2 T) effectively through adaptive scaling

:::{note}
**For EM Students**  
Why Adam works well for electromagnetic field prediction:
1. **Multi-scale field values**: Flux density ranges from mT (far field) to T (magnets). Adam's adaptive learning rates handle this naturally.
2. **Complex loss landscapes**: Electromagnetic constraints (Maxwell's equations, material nonlinearity) create non-convex optimization problems. Momentum helps escape poor local minima.
3. **Large parameter spaces**: Modern CNNs have millions of parameters. Adam's per-parameter adaptation enables efficient training.
4. **Practical advantage**: Adam "just works" with default hyperparameters, reducing manual tuning compared to SGD (which requires careful learning rate scheduling).

**In Section 4c**, we empirically demonstrate Adam achieving 72,000-180,000× speedup over FEA while maintaining <1% prediction error.
:::

**Summary comparison:**

| Optimizer | Momentum | Adaptive LR | Convergence Speed | Hyperparameter Tuning | Best For |
|-----------|----------|-------------|-------------------|----------------------|----------|
| SGD | ✗ | ✗ | Slow | High | Simple problems |
| SGD+Momentum | ✓ | ✗ | Medium | Medium | Well-understood problems |
| RMSprop | ✗ | ✓ | Fast | Low | RNNs, non-stationary |
| Adam | ✓ | ✓ | Fast | Very Low | **CNNs (default choice)** |

In [ ]:
class SimpleNeuralNetwork(nn.Module):
    """
    Simple neural network for optimization demonstration.
    
    Architecture:
    - Input: 2D features (x, x²) 
    - Hidden layers: 2 × 32 neurons with ReLU activation
    - Output: 1D regression (function value)
    
    This mimics the pattern used in EM field prediction where:
    - Input = geometry parameters (slot width, magnet thickness, etc.)
    - Hidden layers = learned electromagnetic relationships
    - Output = predicted field value at spatial location
    """
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=1):
        super(SimpleNeuralNetwork, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.layers(x)

def demonstrate_optimization_algorithms():
    """
    Compare different optimization algorithms on a simple regression task.
    
    Key observations relevant to EM field prediction:
    1. Adam converges faster than SGD (critical for expensive FEA datasets)
    2. Momentum helps SGD escape local minima (important for non-convex loss landscapes)
    3. Adaptive learning rates (Adam, RMSprop) handle varying gradient magnitudes
       (beneficial when field values span multiple orders of magnitude)
    """

    # Generate synthetic data: y = sin(x) + 0.5*cos(2*x) + noise
    # This nonlinear function mimics the complexity of EM field distributions
    np.random.seed(42)
    torch.manual_seed(42)

    n_samples = 1000
    x = torch.linspace(-3, 3, n_samples).unsqueeze(1)
    y_true = torch.sin(x) + 0.5 * torch.cos(2*x)
    y = y_true + 0.1 * torch.randn_like(x)  # Add noise (like FEA mesh discretization error)

    # Create 2D input (x, x²) for more interesting problem
    X = torch.cat([x, x**2], dim=1)

    # Test different optimizers
    optimizers = {
        'SGD': lambda model: torch.optim.SGD(model.parameters(), lr=0.01),
        'SGD + Momentum': lambda model: torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
        'Adam': lambda model: torch.optim.Adam(model.parameters(), lr=0.01),
        'RMSprop': lambda model: torch.optim.RMSprop(model.parameters(), lr=0.01)
    }

    results = {}
    n_epochs = 1000

    # Train with each optimizer
    for opt_name, opt_func in optimizers.items():
        # Create fresh model
        model = SimpleNeuralNetwork(input_dim=2, hidden_dim=32, output_dim=1)
        optimizer = opt_func(model)
        criterion = nn.MSELoss()  # Mean Squared Error - standard for regression

        losses = []

        for epoch in range(n_epochs):
            # Forward pass: compute predictions
            y_pred = model(X)
            loss = criterion(y_pred, y)

            # Backward pass: compute gradients via backpropagation
            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Compute new gradients
            optimizer.step()       # Update weights

            losses.append(loss.item())

            if epoch % 100 == 0:
                print(f"{opt_name:15s} Epoch {epoch:4d}: Loss = {loss.item():.6f}")

        results[opt_name] = {
            'model': model,
            'losses': losses,
            'final_loss': losses[-1]
        }

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Optimization Algorithm Comparison', fontsize=16, fontweight='bold')

    # Plot 1: Training curves (most important for understanding convergence)
    axes[0, 0].set_title('Training Loss Curves', fontweight='bold')
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

    for i, (opt_name, result) in enumerate(results.items()):
        axes[0, 0].plot(result['losses'], label=opt_name, color=colors[i], linewidth=2)

    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss (MSE)')
    axes[0, 0].set_yscale('log')  # Log scale reveals convergence differences
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Final predictions
    axes[0, 1].set_title('Model Predictions vs True Function', fontweight='bold')
    axes[0, 1].scatter(x.numpy(), y.numpy(), alpha=0.3, s=10, color='gray', label='Noisy Data')
    axes[0, 1].plot(x.numpy(), y_true.numpy(), 'k-', linewidth=3, label='True Function')

    for i, (opt_name, result) in enumerate(results.items()):
        with torch.no_grad():
            y_pred = result['model'](X)
            axes[0, 1].plot(x.numpy(), y_pred.numpy(), '--', linewidth=2,
                          color=colors[i], label=opt_name)

    axes[0, 1].set_xlabel('x')
    axes[0, 1].set_ylabel('y')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Final loss comparison
    axes[1, 0].set_title('Final Loss Comparison', fontweight='bold')
    opt_names = list(results.keys())
    final_losses = [results[name]['final_loss'] for name in opt_names]
    bars = axes[1, 0].bar(opt_names, final_losses, color=colors[:len(opt_names)], alpha=0.7)
    axes[1, 0].set_ylabel('Final Loss (MSE)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, loss in zip(bars, final_losses):
        axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                       f'{loss:.6f}', ha='center', va='bottom', fontweight='bold')

    # Plot 4: Convergence speed (epochs to reach 90% of final performance)
    axes[1, 1].set_title('Convergence Speed', fontweight='bold')
    convergence_epochs = []

    for opt_name, result in results.items():
        final_loss = result['final_loss']
        target_loss = final_loss * 10  # 10x final loss as target

        for epoch, loss in enumerate(result['losses']):
            if loss <= target_loss:
                convergence_epochs.append(epoch)
                break
        else:
            convergence_epochs.append(n_epochs)

    bars = axes[1, 1].bar(opt_names, convergence_epochs, color=colors[:len(opt_names)], alpha=0.7)
    axes[1, 1].set_ylabel('Epochs to Convergence')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, epoch in zip(bars, convergence_epochs):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                       f'{epoch}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Print summary
    print("\n" + "="*60)
    print("Optimization Algorithm Summary")
    print("="*60)
    for opt_name, result in results.items():
        final_loss = result['final_loss']
        r2 = r2_score(y_true.numpy(), result['model'](X).detach().numpy())
        print(f"{opt_name:15s}: Final Loss = {final_loss:.6f}, R² = {r2:.4f}")
    
    print("\n" + "="*60)
    print("Why Adam is Preferred for EM Field Prediction CNNs:")
    print("="*60)
    print("[OK] Adaptive learning rates handle varying gradient scales")
    print("    → Field values range from 0.001 T (far field) to 2+ T (magnets)")
    print("[OK] Fast convergence reduces training time")
    print("    → Critical when training on 10,000+ FEA simulations")
    print("[OK] Momentum-like behavior escapes local minima")
    print("    → Important for complex, non-convex loss landscapes")
    print("[OK] Minimal hyperparameter tuning required")
    print("    → Default β₁=0.9, β₂=0.999 work well across geometries")

demonstrate_optimization_algorithms()

### Optimization Algorithm Demonstration

The following demonstration compares four optimization algorithms on a nonlinear regression task. While the problem is simpler than magnetic field prediction, it illustrates key behaviors relevant to training electromagnetic CNNs.

:::{note}
**For EM Students**  
This demonstration reveals three critical insights for field prediction:

1. **Convergence speed**: How quickly the optimizer reaches minimum loss
   - Adam converges 2-5× faster than SGD (critical when training on 10,000+ FEA simulations)
   
2. **Stability**: Oscillations vs. smooth convergence
   - SGD exhibits high variance, Adam shows smooth stable convergence
   - Important for reliable training on expensive FEA datasets
   
3. **Final accuracy**: Quality of converged solution
   - All optimizers can reach similar final accuracy given enough time
   - Adam reaches good solutions fastest with least hyperparameter tuning

In Section 4c, we apply these optimizers to train CNNs on FEA-generated field data, where Adam consistently provides the best trade-off between convergence speed and final accuracy.
:::

## Summary and Connections to Electromagnetic Field Prediction

This section established the machine learning foundations required for developing deep learning models for electromagnetic field prediction. The concepts presented—neural network architectures, optimization algorithms, loss functions, and regularization techniques—form the theoretical basis for all subsequent work in this chapter.

### Core Concepts

**Machine Learning Paradigms**  
Supervised learning for regression maps geometric inputs (motor design parameters) to continuous-valued outputs (magnetic flux density distributions). This paradigm requires labeled training data, which we generate through finite element analysis simulations (Section 3).

**Deep Learning Architectures**  
Hierarchical feature learning in deep networks enables representation of electromagnetic relationships at multiple scales: local field gradients at material boundaries in early layers, regional flux patterns in intermediate layers, and global field structure in deep layers. This multi-scale learning capability motivates the use of convolutional neural networks (Section 4b) over shallow architectures.

**Optimization Theory**  
Four gradient descent variants were analyzed: SGD, SGD with momentum, RMSprop, and Adam. Adam optimizer combines momentum (first moment estimation) and adaptive learning rates (second moment estimation), providing fast convergence with minimal hyperparameter tuning. The empirical demonstration showed Adam's advantages for nonlinear regression tasks, which extend to electromagnetic field prediction where field values span multiple orders of magnitude (0.001-2 T).

**Loss Functions**  
Mean squared error (MSE) quantifies prediction accuracy by aggregating squared errors across all spatial locations in field distributions. For 256×256 pixel field maps (65,536 output values per sample), MSE naturally emphasizes high-flux regions (magnets, airgap) where accurate predictions are critical for torque and loss calculations. Alternative loss functions (MAE, Huber) provide robustness to outliers at the cost of additional hyperparameter tuning.

**Regularization Techniques**  
L2 regularization, dropout, and early stopping prevent overfitting—a critical concern when training on computationally expensive FEA datasets (5,000-15,000 samples). Dropout provides dual benefits: regularization during training and uncertainty quantification during inference (Section 5), enabling estimation of prediction confidence for safety-critical electromagnetic design applications.

### Connection to Chapter Structure

The machine learning fundamentals established here enable the technical developments in subsequent sections:

**Section 3: Data Generation Pipeline**  
Design space sampling strategies (Latin Hypercube Sampling) ensure representative coverage of geometric parameters. FEA simulations provide high-fidelity training labels. Data preprocessing (normalization, augmentation) prepares field distributions for CNN training using the loss functions and optimization algorithms presented here.

**Section 4a: Artificial Neural Network Fundamentals**  
Detailed analysis of activation functions (derivatives, vanishing gradients), backpropagation algorithm (chain rule, gradient computation), and weight initialization strategies. These topics extend the conceptual introduction provided in this section with implementation-level rigor required for training deep networks.

**Section 4b: Convolutional Neural Network Architecture**  
Encoder-decoder CNN architectures for spatial field prediction. Convolutional layers replace fully-connected layers to exploit spatial structure in electromagnetic problems. Dilated convolutions capture long-range field dependencies. Skip connections preserve fine-scale features. Architecture design applies the hierarchical learning principles introduced here to electromagnetic-specific requirements.

**Section 4c: CNN Training and Validation**  
Implementation of training procedures using Adam optimizer, MSE loss, and regularization techniques (dropout, early stopping, L2 weight decay) introduced in this section. Validation on three electromagnetic case studies: air-core coil (linear materials), transformer core (nonlinear saturation), permanent magnet motor (complex geometry, multi-material). Performance metrics quantify generalization to unseen geometries.

**Section 4d: Results and Analysis**  
Quantitative evaluation demonstrating normalized MSE below 1% across all case studies. Computational speedup factors of 72,000-180,000× relative to two-dimensional FEA. Ablation studies isolating the contribution of architectural choices (skip connections, dilated convolutions, dropout) to final prediction accuracy.

**Section 5: Uncertainty Quantification via Bayesian Deep Learning**  
Monte Carlo dropout extends the regularization technique introduced here to uncertainty estimation. Multiple forward passes with dropout active provide prediction distributions rather than point estimates. Confidence intervals enable decision criteria: high-confidence predictions from CNN, low-confidence cases revert to FEA verification. This addresses the practical deployment challenge of automated electromagnetic design.

**Section 6: Physics-Informed Neural Networks (PINNs)**  
Alternative approach incorporating Maxwell's equations directly into loss functions. Residuals of governing PDEs (Ampère's law, magnetic flux continuity) supplement data-driven MSE loss. Physics-informed training reduces required FEA simulations by 50-70% while maintaining prediction accuracy. Hybrid methods combine strengths of data-driven CNNs (Section 4) and physics-constrained PINNs.

### Critical Requirements for Electromagnetic Applications

Three electromagnetic-specific considerations distinguish field prediction from generic regression tasks:

1. **Data efficiency**: FEA simulations require 2-8 hours per geometry. Machine learning models must achieve engineering accuracy (<5% error for global quantities, <1% normalized error for field distributions) with limited training data (5,000-15,000 samples). This constraint motivates transfer learning, physics-informed methods, and architecture choices that exploit domain knowledge.

2. **Physical consistency**: Predictions must satisfy Maxwell's equations approximately, even for geometries outside the training distribution. Pure data-driven methods can generate physically implausible fields (flux discontinuities at material boundaries, violations of source-field relationships). Architectural constraints (Section 4b) and physics-informed losses (Section 6) enforce electromagnetic principles.

3. **Uncertainty quantification**: Unlike image classification where misclassifications cause minor inconvenience, incorrect field predictions lead to motor failures (underestimated saturation, torque overestimation). Deployment in automated design workflows requires confidence estimates to identify when CNN predictions are reliable versus when expensive FEA verification is necessary. Bayesian methods (Section 5) address this safety-critical requirement.

### Methodological Progression

The chapter follows a systematic progression from conventional methods to advanced techniques:

**Foundation (Sections 2-3)**: Machine learning fundamentals and data generation establish baseline capabilities. Supervised learning on FEA-generated training data provides initial surrogate models.

**Core Method (Sections 4a-4d)**: Convolutional neural networks for field distribution prediction. Encoder-decoder architectures with skip connections, trained using Adam optimizer and MSE loss with dropout regularization. Validation demonstrates <1% normalized error and 72,000-180,000× speedup.

**Reliability (Section 5)**: Monte Carlo dropout for uncertainty quantification. Prediction confidence enables safe deployment in automated design, allocating expensive FEA resources to high-uncertainty cases while using fast CNN inference for confident predictions.

**Data Efficiency (Section 6)**: Physics-informed neural networks incorporate Maxwell's equations into training. Governing equation residuals supplement supervised loss, reducing required training data by 50-70% while maintaining accuracy. Hybrid approaches combine data-driven and physics-based paradigms.

This progression addresses the fundamental challenge established in Chapter 1: finite element analysis provides high accuracy but imposes prohibitive computational cost for comprehensive design space exploration. Deep learning surrogates enable evaluation of millions of geometric variants through rapid neural network inference (10-100 milliseconds per geometry), reserving expensive finite element verification for the most promising candidates identified through machine learning-accelerated search.

---

**Next Section**: Section 3 (Data Generation Pipeline) describes the finite element analysis workflow for generating training data, Latin Hypercube Sampling for design space coverage, and preprocessing techniques for preparing field distributions for CNN training.

## Summary and Next Steps

This notebook established the fundamental machine learning concepts necessary for understanding and implementing deep learning models for magnetic field prediction. These principles form the theoretical foundation for all subsequent work in this chapter.

### Key Concepts Covered

**1. Neural Network Architecture**
- Artificial neurons and activation functions (ReLU, sigmoid, tanh)
- Feedforward networks and the universal approximation theorem
- Deep vs. shallow networks: hierarchical feature learning for EM applications

**2. Optimization Algorithms**
- Gradient descent variants: SGD, momentum, RMSprop, Adam
- **Practical finding**: Adam optimizer provides best convergence for EM field prediction
- Learning rate schedules and adaptive methods

**3. Loss Functions**
- Mean Squared Error (MSE) for regression tasks
- Trade-offs between MSE, MAE, and Huber loss
- Design considerations for spatial field prediction

**4. Regularization and Generalization**
- Bias-variance tradeoff in surrogate modeling
- L2 regularization, dropout {cite}`srivastava2014dropout`, and early stopping
- Model selection via cross-validation and random search {cite}`bergstra2012random`

### Connection to Electromagnetic Field Prediction

These ML fundamentals directly enable the technical approach developed in subsequent sections:

**→ Section 3: Data Pipeline** {cite}`goodfellow2016deep`
- Latin Hypercube Sampling for design space coverage
- FEA simulation workflow for generating training data
- Train/validation/test split strategies

**→ Section 4a: ANN Fundamentals**
- Detailed activation function analysis with derivatives
- Backpropagation algorithm for gradient computation
- Weight initialization strategies

**→ Section 4b: CNN Architecture**
- Convolutional layers for spatial field distributions
- Encoder-decoder architectures with skip connections
- Dilated convolutions for long-range electromagnetic interactions

**→ Section 4c: CNN Training**
- Applying Adam optimizer to FEA datasets
- Loss function design for field prediction
- Regularization strategies (dropout, weight decay, early stopping)

**→ Section 5: Uncertainty Quantification**
- Monte Carlo dropout for prediction confidence {cite}`gal2016dropout`
- Bayesian deep learning for EM applications
- Decision criteria: When to trust CNN vs. revert to FEA

**→ Section 6: Physics-Informed Neural Networks**
- Incorporating Maxwell's equations into loss functions
- Reducing data requirements through physics constraints
- Hybrid data-driven and physics-based approaches

### Critical Takeaways for EM Applications

:::{important}
**From General ML to Electromagnetic Surrogates**

1. **Data efficiency matters**: FEA simulations are expensive (2-8 hours each). ML methods must achieve high accuracy with limited training data (5,000-15,000 samples).

2. **Spatial structure is key**: Field distributions are not arbitrary arrays—they obey Maxwell's equations. Convolutional architectures (Section 4b) exploit this spatial structure.

3. **Uncertainty quantification is critical**: Unlike image classification where mistakes are inconsequential, incorrect field predictions can lead to motor failures. Dropout-based uncertainty (Section 5) enables safe deployment.

4. **Physics constraints help**: Pure data-driven methods ignore known electromagnetic laws. Physics-informed approaches (Section 6) incorporate domain knowledge to improve generalization.
:::